# Transformer Language Model Training - Google Colab

This notebook trains a transformer-based language model on TinyShakespeare data using BPE tokenization.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("Using CPU - training will be slower")
print(f"Using device: {device}")

## 1. Install Required Packages

In [ ]:
# Install packages (only needed in Colab)
import subprocess
import sys

packages = ['tokenizers', 'tqdm']
for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed")

## 2. Google Drive Setup (Colab Only)

In [ ]:
# Check if running in Colab
in_colab = 'google.colab' in str(get_ipython())

if in_colab:
    print("Running in Google Colab - Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = '/content/drive/My Drive/LLM_Practice'
else:
    print("Running locally - Using local paths...")
    project_path = '.'
    
print(f"Project path: {project_path}")

## 3. Import Custom Modules and Load Data

In [ ]:
import sys
sys.path.insert(0, project_path)

# Import custom modules
from src.data_loader import TinyShakespeareDataset
from src.embedding import EmbeddingLayer
from src.transformer import DecoderBlock

print("Custom modules imported successfully!")

# Load and prepare data
print("\nLoading TinyShakespeare data...")
dataset = TinyShakespeareDataset(context_length=128, vocab_size=5000)
print(f"Dataset created: {len(dataset)} samples")

# Create dataloaders
batch_size = 32
train_loader, val_loader = dataset.create_dataloaders(batch_size=batch_size, split=0.9)
print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

## 4. Model Configuration

In [ ]:
# Model hyperparameters
D_IN = 256  # Embedding dimension
D_HIDDEN = 1024  # Feed-forward hidden dimension
NUM_HEADS = 8  # Number of attention heads
NUM_BLOCKS = 4  # Number of transformer blocks
CONTEXT_LENGTH = 128  # Sequence length
VOCAB_SIZE = 5000  # Tokenizer vocabulary size
DROPOUT = 0.1  # Dropout rate

# Training hyperparameters
EPOCHS = 5
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

print("Configuration:")
print(f"  D_IN: {D_IN}")
print(f"  D_HIDDEN: {D_HIDDEN}")
print(f"  NUM_HEADS: {NUM_HEADS}")
print(f"  NUM_BLOCKS: {NUM_BLOCKS}")
print(f"  CONTEXT_LENGTH: {CONTEXT_LENGTH}")
print(f"  VOCAB_SIZE: {VOCAB_SIZE}")
print(f"  EPOCHS: {EPOCHS}")
print(f"  LEARNING_RATE: {LEARNING_RATE}")

## 5. Build Model

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_in, d_hidden, num_heads, num_blocks, context_length, dropout=0.1):
        super().__init__()
        self.embedding = EmbeddingLayer(
            vocab_size=vocab_size,
            d_in=d_in,
            max_seq_len=context_length,
            pos_learnable=False,  # Fixed sine/cosine positional encoding
            dropout=dropout
        )
        self.blocks = nn.ModuleList([
            DecoderBlock(d_in, num_heads, d_hidden, dropout)
            for _ in range(num_blocks)
        ])
        self.output_layer = nn.Linear(d_in, vocab_size)
        
    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        logits = self.output_layer(x)
        return logits

# Create model and move to device
model = TransformerModel(
    vocab_size=VOCAB_SIZE,
    d_in=D_IN,
    d_hidden=D_HIDDEN,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    context_length=CONTEXT_LENGTH,
    dropout=DROPOUT
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Summary:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Device: {device}")

## 6. Training Setup

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Training history
history = {
    'train_loss': [],
    'val_loss': []
}

print("Training setup complete!")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Loss: CrossEntropyLoss")

## 7. Training Loop

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch_idx, (inputs, targets) in enumerate(tqdm(train_loader, desc="Training")):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Forward pass
        logits = model(inputs)
        # Reshape for loss computation
        loss = criterion(logits.view(-1, VOCAB_SIZE), targets.view(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for inputs, targets in tqdm(val_loader, desc="Evaluating"):
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = criterion(logits.view(-1, VOCAB_SIZE), targets.view(-1))
            total_loss += loss.item()
    
    return total_loss / len(val_loader)

# Train model
print(f"Starting training for {EPOCHS} epochs...\n")
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")

print("\nTraining complete!")

## 8. Visualize Training Progress

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Train Loss', marker='o')
plt.plot(history['val_loss'], label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=100)
plt.show()

print(f"Training Loss: {history['train_loss'][-1]:.4f}")
print(f"Validation Loss: {history['val_loss'][-1]:.4f}")

## 9. Save Model

In [ ]:
# Save model
model_path = os.path.join(project_path, 'transformer_model.pt')
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

# Save training history
import json
history_path = os.path.join(project_path, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f)
print(f"Training history saved to {history_path}")

## 10. Generate Text

In [ ]:
def generate_text(model, dataset, prompt_text, max_length=200, temperature=1.0, device='cpu'):
    """Generate text from trained model"""
    model.eval()
    tokenizer = dataset.tokenizer
    
    # Tokenize prompt
    tokens = tokenizer.encode(prompt_text).ids
    if len(tokens) > CONTEXT_LENGTH:
        tokens = tokens[-CONTEXT_LENGTH:]
    
    generated = tokens.copy()
    
    with torch.no_grad():
        for _ in range(max_length):
            # Prepare input (keep only last CONTEXT_LENGTH tokens)
            input_tokens = generated[-CONTEXT_LENGTH:] if len(generated) >= CONTEXT_LENGTH else generated
            input_tensor = torch.tensor([input_tokens], device=device)
            
            # Get predictions
            logits = model(input_tensor)
            logits = logits[0, -1, :] / temperature  # Take last token, apply temperature
            
            # Sample next token
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            generated.append(next_token)
            
            # Stop if end of sequence
            if next_token == tokenizer.token_to_id("[PAD]"):
                break
    
    # Decode generated text
    generated_text = tokenizer.decode(generated[len(tokens):])
    return generated_text

# Generate text
print("Generating text from trained model...\n")
prompt = "The king said"
generated = generate_text(model, dataset, prompt, max_length=150, temperature=0.8, device=device)
print(f"Prompt: {prompt}")
print(f"Generated: {prompt} {generated}")
print("\n" + "="*50 + "\n")

# Generate more examples
for i in range(2):
    prompt = "To be or not to be"
    generated = generate_text(model, dataset, prompt, max_length=100, temperature=0.9, device=device)
    print(f"Example {i+1}:")
    print(f"{prompt} {generated}\n")

In [1]:
print()